# MR-FBP — Minimum Residual Filtered Backprojection

Reproduction and extension of Pelt & Batenburg, *"Improving filtered backprojection
reconstruction by data-dependent filtering"*, IEEE Trans. Image Processing, 2014.

**Before running: Runtime → Change runtime type → T4 GPU.**

This notebook is only a launcher. All real code lives in `.py` files in the repo,
which is cloned from GitHub — so every result is traceable to a commit.

**Working loop:** edit locally → `git commit && git push` → re-run *Setup* below (`git pull`) → run an experiment.

---

### The idea in one box

FBP is **linear in its filter**. Convolution commutes, so

$$\mathrm{FBP}_h(p) = W^{T} C_h\, p = W^{T} C_p\, h$$

i.e. the reconstruction is a *linear map of the filter*. So instead of picking Ram-Lak
by hand, **solve** for the filter that minimises the projection residual:

$$h^* = \arg\min_h \lVert p - W W^{T} C_p h \rVert^2 \quad\Longrightarrow\quad A_p h = p,\;\; A_p = W W^{T} C_p$$

Exponential binning cuts the unknowns to $O(\log N_d)$ (~10 for $N_d=256$), so the
least-squares solve is **direct, iteration-free and parameter-free**. The reconstruction
is then a plain FBP with the computed filter: near-FBP speed, near-algebraic quality.

## Setup

In [ ]:
!pip install -q astra-toolbox scikit-image
import astra; astra.test()   # expects a GPU; 'Ok' on all lines

In [ ]:
REPO = 'https://github.com/maradimitriu/MRFBP.git'

import os, subprocess
if not os.path.isdir('/content/mrfbp'):
    subprocess.run(['git', 'clone', REPO, '/content/mrfbp'], check=True)
%cd /content/mrfbp
!git pull --ff-only
!git log -1 --oneline    # <- the exact commit that produced everything below

from IPython.display import Image, display
def show(name):
    display(Image(filename=f'results/{name}.png'))

## Verification

Two independent gates. If the first passes and the second fails, the problem is ASTRA
or the geometry — **not** the algorithm. That separation is why they are separate scripts.

In [ ]:
# Builds W densely at 32x32 (so W^T is the EXACT transpose) and checks the three
# claims the paper rests on:
#   [1] FBP is linear in the filter:  C_h p == C_p h
#   [2] build_Ap() really computes    W W^T C_p
#   [3] the least-squares filter beats any fixed filter on projection residual
# No ASTRA involved -- this also runs on a laptop with no GPU.
!python tests/test_core.py

In [ ]:
# ASTRA plumbing: fp/bp round-trip, our FBP vs ASTRA's own FBP, first MR-FBP.
# The printed best-fit scale must be ~1.0, or the pi/N_theta normalisation is wrong.
!python scripts/smoke_test.py
show('smoke_test')

## Experiments

Each script prints all of its parameters at startup and accepts `--help`.
Defaults are the fast settings ($N=256$); pass `--n 1024` for the paper's resolution.

| script | what it shows | paper |
|---|---|---|
| exp1 | quality vs number of projections, 6 methods, 3 phantoms | Figs. 7, 8 |
| exp2 | robustness to Poisson noise | Figs. 10, 11 |
| exp3 | reconstruction time | Fig. 9 |
| exp4 | exponential binning: cost vs quality, sweeping `n_l` | Sec. VII-D |
| exp5 | the computed filters, in Fourier space | Fig. 16 |
| exp6 | MR-FBP$_{GM}$: a gradient prior, **+ our λ sweep** | Figs. 14, 15 |
| **exp7** | **own:** does the filter basis matter? | — |
| **exp8** | **own:** how data-dependent is the filter really? | — |
| **exp9** | **own:** regime map — where MR-FBP wins and loses | — |

### exp1 — quality vs number of projections  *(~15 min)*
MR-FBP should beat every fixed-filter FBP at every projection count, sit above SIRT,
and fall monotonically. On clean data it does, on all three phantoms.

In [ ]:
!python experiments/exp1_projections.py --n 256
show('exp1_mae'); show('exp1_images')

### exp2 — Poisson noise  *(~4 min)*
FBP-RL collapses under noise (the ramp amplifies exactly where photon noise lives);
MR-FBP tracks SIRT — **with no regulariser at all**.

In [ ]:
!python experiments/exp2_noise.py
show('exp2_noise'); show('exp2_images')

### exp3 — reconstruction time  *(~3 min)*
MR-FBP ≈ 10–18× faster than SIRT-200 (it performs $O(\log N_d)$ projections, not $O(N_d)$).

⚠️ Colab's GPU is **shared and throttled**, so absolute timings are not reproducible.
The script does a global warm-up and reports the *minimum* of 5 repeats (on a shared GPU
every sample is the true cost plus non-negative contention noise, so the minimum is the
least contaminated estimate). Report the hardware alongside these numbers.

In [ ]:
!python experiments/exp3_timing.py
show('exp3_timing')

### exp4 — exponential binning  *(~3 min)*
Quality is roughly flat while time varies ~87×: the binning is nearly free.

But note the **residual falls while the MAE rises** over $N_b \in [10, 136]$ — the objective
MR-FBP minimises diverges from the thing we care about. And `n_l` is a *hidden regularisation
parameter*, which qualifies the method's "parameter-free" selling point.

In [ ]:
!python experiments/exp4_binning.py
show('exp4_binning')

import numpy as np
d = np.load('results/exp4_binning.npz')
for k in ('n_bins', 'res', 'mae', 'ssim'):
    print(f'{k:7s}', np.round(d[k], 4))

### exp5 — the computed filters  *(~2 min)*
**The figure that explains everything else.** Panel (a) is raw gain — the filter actually
applied — against $(\pi/N_\theta)\,h_{RL}$.

- every computed filter sits **below** Ram-Lak at high frequency (~40% of its gain at Nyquist);
- **noise deepens the roll-off** by a further ~25% — nobody told it to do that. There is no
  regulariser in MR-FBP; minimising the projection residual *discovers* noise suppression;
- more angles push the filter **toward** Ram-Lak (40% → 57% of its Nyquist gain from 64 → 128
  projections). That drift is the mechanism behind exp8 and exp9.

In [ ]:
!python experiments/exp5_filters.py
show('exp5_filters')

### exp6 — MR-FBP$_{GM}$ and the λ heuristic  *(~8 min)*
The gradient prior costs **no extra projections** (the reconstructions are already in hand).

The paper's $\lambda = 27 + 1600/I_0$ is presented as a universal "reasonable choice", but λ
balances two terms whose relative magnitudes depend on image size, detector count and data
scaling — it is **not scale-invariant**. At $N=256$ it over-regularises by ~27× and
MR-FBP$_{GM}$ stops responding to the data (the flat curve).

With λ swept properly, MR-FBP$_{GM}$ is **4.5× more accurate than SIRT-200** at $I_0 = 64$.

In [ ]:
!python experiments/exp6_gradmin.py
show('exp6_gradmin'); show('exp6_lambda'); show('exp6_images')

### exp7 — does the filter basis matter?  *(own, ~4 min)*
The paper's conclusion flags this as future work — *"other bases ... can be used however,
which might enable us to reduce computation time even further, or improve reconstruction
quality"* — and never tries. We compare four bases **at matched numbers of unknowns**.

Answer: exponential ≈ Gaussian-RBF $\gg$ equidistant ≈ DCT (**~5×**). Since Gaussian RBFs
placed *at the exponential centres* match exponential bins exactly, it is the **exponential
placement** that matters, not the piecewise-constant shape. Exponential binning is essential,
not merely convenient.

In [ ]:
!python experiments/exp7_bases.py
show('exp7_bases')

### exp8 — how data-dependent is the filter really?  *(own, ~6 min)*
Learn $h^*$ on a source problem, apply it to a target, and measure the penalty.
The diagonal is true MR-FBP; off-diagonal is the transfer cost.

Two findings the paper never measures:

1. **On noisy data the native filter is the *wrong* filter.** A filter learned from *fewer*
   projections beats it by up to **2.3×**.
2. **The filter barely depends on the object.** Swapping the phantom costs ~5%; swapping
   the projection count costs ~86%. The filter is a function of *geometry and noise*, not of
   what you are imaging — which corrects the paper's Fig. 16 claim that "there is no single
   filter that is ideal for every problem".

In [ ]:
!python experiments/exp8_transfer.py
show('exp8_transfer')

import numpy as np
d = np.load('results/exp8_transfer.npz')
M, L = d['matrix'], d['labels']
print('  source \\ target ' + ' '.join(f'{l[:10]:>10s}' for l in L))
for i, l in enumerate(L):
    print(f'{l[:16]:>16s} ' + ' '.join(f'{v:10.4f}' for v in M[i]))

### exp9 — regime map: where does MR-FBP win?  *(own, ~20 min)*
The headline. The paper evaluates noise at **one** projection count ($N_\theta = 64$) and
concludes MR-FBP beats static-filter FBP *"for every noise level"*. Sweeping the full
$(N_\theta, I_0)$ plane shows that is regime-specific:

- MR-FBP is **beaten by FBP-Hann on 29% of the grid** (worst case **2.80×** at $N_\theta=192$, $I_0=64$);
- whenever $I_0 \lesssim 10^4$, **more projections make MR-FBP worse** — its error is
  non-monotonic in the amount of data.

**Mechanism** (from exp5): MR-FBP's noise robustness is not a property of the method, it is an
artefact of **data scarcity**. With few angles it *cannot* explain high frequencies, so the
least-squares suppresses them — which happens to be right under noise. With many angles it
becomes confident enough to amplify them, which is exactly wrong. And because MR-FBP is
*parameter-free*, there is **no knob to turn** to fix it — which is precisely what
MR-FBP$_{GM}$ restores (exp6).

Pass `--no-sirt` to run ~3× faster.

In [ ]:
!python experiments/exp9_regime.py
show('exp9_regime'); show('exp9_slices')

## Reproduce everything, and save it

Run this once before submitting, so that **every figure comes from one commit**. Mixing
results from different code versions is the classic way to end up with a paper whose
figures quietly disagree with each other.

In [ ]:
!rm -rf results && ./run_all.sh

In [ ]:
# results/ lives on the Colab VM and is WIPED when the runtime dies. Download it.
!zip -qr /content/results.zip results
from google.colab import files; files.download('/content/results.zip')